# Neutron

In [4]:
import h5py

file_data = h5py.File('neutron/Ac225.h5', 'r')

# Menampilkan seluruh hierarki/struktur di dalam file
print("Struktur isi file:")
file_data.visit(print)

Struktur isi file:
Ac225
Ac225/energy
Ac225/energy/0K
Ac225/energy/1200K
Ac225/energy/2500K
Ac225/energy/250K
Ac225/energy/294K
Ac225/energy/600K
Ac225/energy/900K
Ac225/fission_energy_release
Ac225/fission_energy_release/betas
Ac225/fission_energy_release/delayed_neutrons
Ac225/fission_energy_release/delayed_photons
Ac225/fission_energy_release/fragments
Ac225/fission_energy_release/neutrinos
Ac225/fission_energy_release/prompt_neutrons
Ac225/fission_energy_release/prompt_photons
Ac225/fission_energy_release/q_prompt
Ac225/fission_energy_release/q_recoverable
Ac225/kTs
Ac225/kTs/1200K
Ac225/kTs/2500K
Ac225/kTs/250K
Ac225/kTs/294K
Ac225/kTs/600K
Ac225/kTs/900K
Ac225/reactions
Ac225/reactions/reaction_002
Ac225/reactions/reaction_002/0K
Ac225/reactions/reaction_002/0K/xs
Ac225/reactions/reaction_002/1200K
Ac225/reactions/reaction_002/1200K/xs
Ac225/reactions/reaction_002/2500K
Ac225/reactions/reaction_002/2500K/xs
Ac225/reactions/reaction_002/250K
Ac225/reactions/reaction_002/250K/xs
Ac

Dalam struktur data HDF5 OpenMC, **`product_0`** dan **`product_1`** merepresentasikan **jenis partikel sekunder yang berbeda** yang dipancarkan dari satu reaksi nuklir yang sama.

Untuk memahami konteks fisisnya, kita perlu tahu bahwa `reaction_016` (MT=16 dalam standar ENDF) adalah sandi untuk **reaksi $(n, 2n)$**. Ini adalah reaksi di mana satu neutron menabrak inti atom target, lalu menyebabkan **dua neutron** terlempar keluar.

Berikut adalah rincian perbedaan keduanya:

### 1. `product_0` (Neutron)

Secara bawaan (*default*), produk pertama (`product_0`) dari reaksi yang diinisiasi oleh neutron hampir selalu adalah **Neutron** itu sendiri.

* Meskipun reaksi $(n, 2n)$ menghasilkan 2 buah neutron, OpenMC tidak membaginya menjadi produk 0 dan produk 1.
* Sebagai gantinya, OpenMC hanya mendefinisikan *jenis* partikelnya (neutron) di `product_0`, lalu mengatur jumlah partikel yang keluar di dalam *path* **`product_0/yield`**. (Jika Anda membongkar data `yield` ini, Anda akan menemukan nilai konstan `2.0`).
* *Path* `distribution_0/...` di bawahnya menyimpan informasi tentang bagaimana kedua neutron tersebut membagi energi (`energy_out`) dan ke arah mana mereka terpental (`mu`).

### 2. `product_1` (Foton / Sinar Gamma)

Lalu, partikel apa yang menjadi `product_1`? Dalam mayoritas data nuklir, produk kedua ini adalah **Foton (Sinar Gamma)**.

* Saat reaksi $(n, 2n)$ terjadi, inti atom target (misalnya $Ac^{225}$) kehilangan satu neutron dan berubah menjadi isotop lain ($Ac^{224}$). Inti sisa ini biasanya berada dalam keadaan tereksitasi (kelebihan energi).
* Untuk kembali ke keadaan stabil, inti sisa tersebut memancarkan energi berlebihnya dalam bentuk radiasi gamma.
* OpenMC harus memisahkan data foton ini ke dalam folder `product_1` karena foton memiliki spektrum energi emisi dan distribusi sudut yang sifat fisisnya sama sekali berbeda dari hamburan neutron.

---

### Cara Membuktikannya Lewat Kode

Anda sebenarnya bisa mengonfirmasi jenis partikel apa yang ada di dalam masing-masing "folder" produk tersebut. OpenMC menyimpannya sebagai **atribut** pada grup `product_0` dan `product_1` itu sendiri (bukan pada dataset di dalamnya).

Jika Anda menjalankan skrip Python Anda sebelumnya dan menargetkan *path* grup utamanya, Anda bisa melihat nama partikelnya:


In [23]:
import h5py

filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # Mengecek atribut jenis partikel pada product_0
    partikel_0 = f['Ac225/reactions/reaction_016/product_0'].attrs.get('particle')
    print(f"Product 0 adalah: {partikel_0}")  # Hasilnya akan b'neutron'
    
    # Mengecek atribut jenis partikel pada product_1
    partikel_1 = f['Ac225/reactions/reaction_016/product_1'].attrs.get('particle')
    print(f"Product 1 adalah: {partikel_1}")  # Hasilnya akan b'photon'

Product 0 adalah: b'neutron'
Product 1 adalah: b'photon'


Ini adalah penemuan yang sangat bagus! Data yang Anda bongkar ini secara sempurna mengonfirmasi apa yang kita bahas pada pertanyaan sebelumnya.

Anda sedang melihat data **`yield`** (jumlah partikel yang dihasilkan) untuk **`product_0`** (neutron) dari **`reaction_016`** (reaksi $(n, 2n)$).

Mari kita bedah maknanya berdasarkan output skrip Anda:

### 1. Mengapa Tipenya `Polynomial`?

Sama seperti saat Anda memeriksa data `fission_energy_release/betas` sebelumnya, atribut ini memberi tahu OpenMC bahwa nilai *yield* tidak disimpan dalam bentuk tabel energi yang panjang, melainkan dievaluasi menggunakan rumus polinomial:

$$Yield(E) = C_0 + C_1 E + C_2 E^2 + \dots$$

### 2. Mengapa Datanya Hanya `[2.]`?

Anda bisa melihat bahwa *shape*-nya adalah `(1,)` yang berarti array tersebut hanya memiliki **satu elemen tunggal**, yaitu angka `2.0`.

Dalam konteks persamaan polinomial OpenMC, jika array hanya memiliki satu elemen, itu berarti angka tersebut adalah **konstanta utamanya ($C_0$)**, sedangkan koefisien lainnya ($C_1, C_2,$ dst) dianggap nol.

Jadi, rumus matematikanya menjadi sangat sederhana:


$$Yield(E) = 2.0$$

### Makna Fisisnya

Data ini secara harfiah menerjemahkan hukum fisika reaksi $(n, 2n)$ ke dalam bahasa mesin.

Rumus $Yield(E) = 2.0$ menginstruksikan OpenMC: *"Tidak peduli berapa pun energi neutron insiden ($E$) yang memicu reaksi ini, jumlah neutron baru yang harus kamu lahirkan (yield) di dalam simulasi selalu **mutlak 2 buah**."*

---

**Sebagai perbandingan:**
Jika nanti Anda memeriksa data *yield* neutron untuk reaksi fisi murni (misalnya MT=18), Anda mungkin akan melihat array yang lebih panjang atau atribut tipe `Tabulated1D`. Hal ini karena jumlah neutron yang keluar dari fisi ($\nu$ atau *nu-bar*) bergantung pada energi neutron yang menabraknya (semakin tinggi energi tabrakan, jumlah neutron fisi yang keluar bisa lebih dari 2.4, 2.5, dst). Namun untuk reaksi ambang seperti $(n, 2n)$ atau $(n, 3n)$, nilainya selalu berupa konstanta bilangan bulat `2.0` atau `3.0`.

In [25]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_0/yield'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/reactions/reaction_016/product_0/yield

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (1,)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'type' = b'Polynomial'

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 1
--> Seluruh Data   : [2.]


Menarik sekali, bukan? Perbedaan ini justru memperlihatkan kontras yang sangat jelas antara karakteristik fisik **Neutron** dan **Foton (Sinar Gamma)** dalam reaksi nuklir.

Mari kita bedah mengapa *yield* untuk `product_1` (foton) ini berformat **`Tabulated1D`** dan nilainya berubah-ubah (dinamis), sangat berbeda dengan *yield* neutron yang konstan `[2.]`.

---

### 1. Perbedaan Karakteristik Fisis (Sebab Utama)

* **Neutron (`product_0`):** Reaksi ini adalah $(n, 2n)$. Secara mekanika kuantum dan hukum kekekalan jumlah nukleon, jumlah neutron yang keluar **harus mutlak 2**. Tidak bisa 1.5 atau 2.5 neutron. Itulah mengapa nilainya konstan polinomial `2.0`.
* **Foton (`product_1`):** Sinar gamma dipancarkan ketika inti atom sisa ($Ac^{224}$) yang dalam keadaan tereksitasi melepaskan energi berlebihnya untuk kembali ke keadaan dasar (*ground state*).

Jumlah foton yang dipancarkan (multiplisitas gamma) **sangat bergantung pada seberapa besar energi neutron insiden yang menabraknya**.

* Jika energi neutron datang pas-pasan di ambang batas ($\sim 6.69\text{ MeV}$), inti atom sisa hanya sedikit tereksitasi, sehingga hanya memancarkan sekitar **1.9 foton**.
* Semakin tinggi energi neutron yang menabrak (misal naik ke $15\text{ MeV}$), inti sisa akan menjadi sangat tidak stabil dan mengalami eksitasi tingkat tinggi. Untuk membuang energi yang sangat besar itu, ia harus memancarkan lebih banyak foton secara bertahap (kaskade gamma), sehingga *yield*-nya melonjak hingga mencapai sekitar **6.5 foton**.

---

### 2. Membaca Struktur Datanya `(2, 28)`

Karena nilainya bergantung pada energi, OpenMC tidak bisa menggunakan rumus polinomial sederhana satu angka. Ia harus menyimpannya dalam bentuk tabel satu dimensi (`Tabulated1D`) dengan ukuran matriks `(2, 28)`:

* **Baris Pertama (Sumbu $x$ - Energi Insiden dalam eV):** Dimulai dari $6.69771 \times 10^6\text{ eV}$ ($6.69\text{ MeV}$) hingga $2.00000 \times 10^7\text{ eV}$ ($20\text{ MeV}$). Ada tepat 28 titik sampel energi.
* **Baris Kedua (Sumbu $y$ - Yield Foton):**
Nilai *yield* aktual yang berpasangan dengan energi di atasnya. Anda bisa melihat tren nilainya:
* Pada $6.69\text{ MeV} \rightarrow \text{Yield} = \mathbf{1.94212}$
* Nilainya naik terus seiring bertambahnya energi...
* Pada sekitar $15.5\text{ MeV} \rightarrow \text{Yield}$ mencapai puncaknya di angka $\mathbf{6.56924}$
* Kemudian sedikit melandai turun ke angka **6.25** pada energi $20\text{ MeV}$.

In [ ]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_1/yield'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Dalam fisika nuklir dan algoritma simulasi Monte Carlo (seperti OpenMC), simbol **$\mu$ (mu)** merepresentasikan **kosinus dari sudut hamburan atau emisi partikel sekunder** terhadap arah gerak aslinya.

Secara matematis, nilainya adalah $\mu = \cos(\theta)$, di mana $\theta$ adalah sudut defleksi partikel.

Karena ini adalah nilai kosinus, rentang nilai $\mu$ selalu berada di antara **-1 hingga 1**:

* **$\mu = 1$ ($\theta = 0^\circ$)**: Partikel sekunder melaju lurus ke depan mengikuti arah neutron penumbuk (*forward emission*).
* **$\mu = 0$ ($\theta = 90^\circ$)**: Partikel terpental tegak lurus dari arah asalnya.
* **$\mu = -1$ ($\theta = 180^\circ$)**: Partikel terpental persis berbalik arah ke belakang (*backscattering*).

### Membaca Matriks `(3, 8212)` Anda

Sama seperti data distribusi energi sebelumnya, OpenMC menggabungkan distribusi sudut dari ribuan titik energi insiden ke dalam satu matriks memori yang panjang. Pola array di tangkapan layar Anda sangat menarik dan menceritakan kondisi fisis yang spesifik.

Berikut adalah makna dari ketiga baris tersebut:

**Baris 1: Nilai $\mu$ (Sumbu $x$)**
Anda melihat pola berulang `[-1.  0.  1.]`. Ini adalah rentang sudut (dari belakang, samping, hingga depan).

**Baris 2: *Probability Density Function* / PDF**
Nilainya adalah `[0.5  0.5  0.5]`. Karena nilai probabilitasnya **konstan rata** (0.5) di seluruh rentang sudut, ini menandakan bahwa distribusi sudut untuk reaksi ini bersifat **Isotropik**. Artinya, partikel hasil reaksi memiliki peluang yang persis sama besar untuk terlempar ke segala arah.

**Baris 3: *Cumulative Distribution Function* / CDF**
Nilainya adalah `[0.  0.5  1.]`. Ini adalah hasil integral dari PDF di atas (luas area di bawah kurva probabilitas). Nilai ujungnya pasti selalu berujung pada angka `1.0`.

### Bagaimana OpenMC Menggunakan Data Ini?

Dalam internal C++ OpenMC, saat melakukan pelacakan partikel (*particle tracking*), program secara konstan harus menentukan ke mana arah partikel bergerak selanjutnya di dalam geometri model.

Ketika reaksi terjadi, OpenMC akan menghasilkan sebuah **angka acak (*random number*) dari 0 hingga 1**. Angka acak tersebut akan dicocokkan ke **Baris 3 (CDF)**. Setelah posisinya ditemukan, OpenMC menarik garis ke atas untuk mengambil nilai $\mu$ di **Baris 1**, lalu menggunakan matriks rotasi 3D untuk mengubah vektor arah gerak partikel tersebut sebelum melanjutkan *random walk* ke sel geometri berikutnya.

In [21]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_0/distribution_0/mu'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/reactions/reaction_016/product_0/distribution_0/mu

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (3, 8212)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Tidak ada atribut yang menempel pada dataset ini.

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 3
--> Seluruh Data   : [[-1.   0.   1.  ... -1.   0.   1. ]
 [ 0.5  0.5  0.5 ...  0.5  0.5  0.5]
 [ 0.   0.5  1.  ...  0.   0.5  1. ]]


Anda baru saja menemukan salah satu teknik optimasi paling penting di dalam struktur memori OpenMC! Kedua atribut ini (`offsets` dan `n_discrete_lines`) adalah cara OpenMC membaca **ratusan tabel probabilitas yang digabung menjadi satu**.

Mari kita hubungkan dengan data Anda sebelumnya. Pada *path* sebelumnya, Anda melihat ada **28 titik energi neutron datang** (dari 6,69 MeV hingga 20 MeV).

Untuk setiap 1 titik energi datang tersebut, partikel sekunder (produk reaksi) yang keluar akan memiliki kurva probabilitas energinya (*spectrum*) masing-masing. Daripada membuat 28 tabel terpisah (yang lambat untuk dibaca komputer), OpenMC menyambung ke-28 tabel tersebut menjadi **satu array raksasa** sepanjang 1280 kolom.

Berikut adalah fungsi dari masing-masing atribut untuk membaca array raksasa tersebut:

### 1. `offsets` (Penunjuk Indeks / Pembatas)

Karena 28 tabel tadi disambung menjadi satu, komputer butuh "penanda buku" untuk tahu di mana tabel untuk energi datang ke-1 berakhir, dan tabel energi datang ke-2 dimulai. Itulah fungsi `offsets`.

Nilai `offsets = [0, 3, 9, 19, 34, 54, ...]` berarti:

* Tabel probabilitas untuk energi datang ke-1 (6,69 MeV) **dimulai pada indeks 0**.
* Tabel probabilitas untuk energi datang ke-2 (7,0 MeV) **dimulai pada indeks 3**. (Artinya tabel pertama hanya punya 3 titik data: indeks 0, 1, dan 2).
* Tabel probabilitas untuk energi datang ke-3 (7,5 MeV) **dimulai pada indeks 9**. (Artinya tabel kedua punya 6 titik data: dari indeks 3 sampai 8).
* ... dan seterusnya untuk ke-28 energi referensi.

### 2. `n_discrete_lines` (Garis Energi Diskrit)

Dalam fisika nuklir, partikel sekunder yang keluar dari suatu reaksi bisa memiliki spektrum energi yang **kontinu** (tersebar halus) atau **diskrit** (keluar pada angka energi yang spesifik/puncak tajam, seperti sinar gamma).

Terkadang, sebuah distribusi energi memiliki gabungan keduanya (beberapa garis energi pasti, diikuti oleh spektrum kontinu). Atribut `n_discrete_lines` memberi tahu OpenMC: *"Dari titik mulai (`offset`) tabel ini, berapa banyak data pertama yang merupakan energi diskrit?"*

* Karena nilai di data Anda adalah `[0 0 0 0 0 0 ...]`, ini berarti untuk reaksi ini (MT=16 atau reaksi $(n,2n)$), **tidak ada partikel yang keluar dengan energi diskrit**. Seluruh distribusinya murni merupakan spektrum kontinu.

---

### Membaca Matriks `(5, 1280)`

Sebagai tambahan informasi, jika Anda penasaran mengapa ukuran datanya adalah 5 baris dan 1280 kolom (`(5, 1280)`), kelima baris tersebut memiliki makna baku di dalam *source code* OpenMC:

* **Baris ke-1:** $E_{out}$ (Energi partikel yang keluar).
* **Baris ke-2:** *Probability Density Function* / PDF (Nilai probabilitas fisis pada titik energi tersebut).
* **Baris ke-3:** *Cumulative Distribution Function* / CDF (Nilai probabilitas kumulatif, bisa dilihat nilainya selalu naik dan berujung di angka `1.000`).
* **Baris ke-4:** Kode Interpolasi (Angka `2` berarti OpenMC menggunakan interpolasi linear).
* **Baris ke-5:** Indeks acuan (biasanya digunakan di internal OpenMC untuk melacak posisi di memori).

In [20]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_0/distribution_0/energy_out'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/reactions/reaction_016/product_0/distribution_0/energy_out

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (5, 1280)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'interpolation' = [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
--> Atribut: 'n_discrete_lines' = [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
--> Atribut: 'offsets' = [   0    3    9   19   34   54   79  109  144  183  224  268  314  363
  414  467  521  577  635  694  754  816  879  943 1009 1076 1143 1211]

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 5
--> Seluruh Data   : [[0.00000000e+00 5.00000000e-01 1.00000000e+00 ... 1.30000000e+07
  1.32000000e+07 1.32429300e+07]
 [0.00000000e+00 2.00000000e+00 0.00000000e+00 ... 1.53322100e-08
  1.66941800e-09 0.00000000e+00]
 [0.00000000e+00 5.00000000e-01 1.00000000e+00 ... 9.98264003e-01
  9.99964166e-01 1.00000000e+00]
 [2.00000000e+00 2.000000

Atribut interpolasi yang Anda temukan kali ini secara prinsip **sama persis** dengan yang sebelumnya, hanya saja format penyimpanannya sedikit berbeda. OpenMC menggabungkan informasi *breakpoints* dan kode interpolasi ke dalam satu buah matriks (array 2D) yang ringkas.

Mari kita bedah nilai atribut `'interpolation' = [[28], [2]]`:

Format array 2D ini terbagi menjadi dua baris:

* **Baris Pertama `[28]`:** Ini adalah nilai **breakpoints** (titik batas).
* **Baris Kedua `[2]`:** Ini adalah **kode interpolasi** (berdasarkan standar ENDF-6).

### Hubungan Atribut dengan Data Aktual

Jika Anda melihat bagian **Properti Dataset Basis**, ukuran datanya adalah `(28,)` yang berarti ada tepat 28 titik data (mewakili rentang energi dari 6,69 MeV hingga 20 MeV).

1. Karena *breakpoint*-nya adalah **28**, ini menginstruksikan OpenMC bahwa aturan interpolasi tersebut berlaku mulai dari data pertama hingga data ke-28 (yang berarti berlaku untuk **seluruh** panjang array tersebut).
2. Karena kodenya adalah **2**, aturan yang digunakan untuk seluruh array tersebut adalah **Linear-Linear**.

### Makna Fisis dalam Konteks Path Tersebut

Path yang Anda periksa adalah `.../product_0/distribution_0/energy`. Ini adalah grid energi neutron yang menabrak target (energi insiden).

Dataset ini berfungsi sebagai sumbu-$x$ referensi untuk menghitung distribusi energi atau sudut partikel sekunder (produk reaksi). Jika di tengah simulasi terdapat neutron dengan energi **7,25 MeV** (yang tidak tertulis di tabel, karena posisinya berada di celah antara `7.000.000` dan `7.500.000`), OpenMC akan menggunakan interpolasi linear (menarik garis lurus) di antara kedua titik tersebut untuk menentukan distribusi partikel sekundernya secara akurat.

In [19]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_0/distribution_0/energy'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/reactions/reaction_016/product_0/distribution_0/energy

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (28,)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'interpolation' = [[28]
 [ 2]]

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 28
--> 5 Data Pertama : [6697710. 7000000. 7500000. 8000000. 8500000.]
--> 5 Data Terakhir: [18000000. 18500000. 19000000. 19500000. 20000000.]


Ketiga atribut tersebut (`type`, `interpolation`, dan `breakpoints`) adalah cara OpenMC (yang mengadopsi standar format data nuklir ENDF-6) untuk mendefinisikan **bagaimana cara membaca dan menginterpolasi kurva titik-titik data**.

Sebelum membahas atributnya, mari kita pahami dulu datanya. Di bagian `[3] NILAI DATA AKTUAL`, Anda memiliki array `(2, 2)`:

* **Baris Pertama (Sumbu $x$):** `[6.69771e+06, 2.00000e+07]` $\rightarrow$ Ini adalah rentang energi neutron datang, dari sekitar 6,69 MeV hingga 20 MeV.
* **Baris Kedua (Sumbu $y$):** `[1.0, 1.0]` $\rightarrow$ Ini adalah nilai *applicability* (probabilitas). Nilai 1.0 berarti 100%.

Jadi, data ini secara fisik mengatakan: "Distribusi partikel sekunder ini 100% berlaku untuk neutron datang dengan energi 6,69 MeV hingga 20 MeV".

Berikut adalah makna dari masing-masing atribut yang mengatur data tersebut:

### 1. `type = b'Tabulated1D'`

Atribut ini memberi tahu OpenMC bahwa dataset di bawahnya harus diperlakukan sebagai **fungsi tabel satu dimensi** ($y = f(x)$).
Dalam OpenMC, kelas `Tabulated1D` akan mengubah baris pertama array sebagai nilai $x$ (biasanya energi) dan baris kedua sebagai nilai $y$ (probabilitas, cross-section, yield, dll).

### 2. `interpolation = [2]`

Atribut ini mendefinisikan **aturan matematika untuk mencari nilai di antara titik-titik data**.
Kode angka di sini mengikuti standar evaluasi data nuklir internasional (ENDF-6 format). Berikut adalah arti kode interpolasi ENDF yang paling umum:

* **1** : Histogram (Konstan)
* **2** : **Linear-Linear** ($y$ diinterpolasi secara linear terhadap $x$)
* **3** : Linear-Log ($y$ linear, $x$ logaritmik)
* **4** : Log-Linear ($y$ logaritmik, $x$ linear)
* **5** : Log-Log (Keduanya logaritmik)

Karena nilainya `[2]`, OpenMC akan menggunakan interpolasi garis lurus (linear) biasa jika program membutuhkan nilai probabilitas pada energi di antara 6,69 MeV dan 20 MeV.

### 3. `breakpoints = [2]`

Atribut ini mendefinisikan **kapan aturan interpolasi berubah**.
Pada data nuklir yang sangat kompleks, terkadang evaluator menggunakan interpolasi Log-Log (5) untuk rentang energi rendah, lalu berubah menjadi Linear-Linear (2) di energi tinggi dalam satu dataset yang sama.

* `breakpoints` menyimpan nomor indeks (posisi) di mana suatu aturan interpolasi berakhir.
* Karena dataset Anda sangat pendek (hanya memiliki 2 kolom data), `breakpoints = [2]` berarti aturan `interpolation = [2]` (Linear-Linear) berlaku dari awal data sampai titik ke-2 (yang merupakan akhir dari data tersebut).

**Kesimpulan Singkat:**
Atribut-atribut tersebut secara kolektif menginstruksikan OpenMC: *"Baca array ini sebagai kurva 1D ($x, y$). Jika simulasi membutuhkan nilai $y$ di suatu titik $x$ yang tidak ada persis di dalam tabel, tariklah garis lurus (interpolasi linear) dari titik pertama sampai titik terakhir untuk menemukan nilainya."*

In [18]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/reactions/reaction_016/product_0/distribution_0/applicability'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/reactions/reaction_016/product_0/distribution_0/applicability

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (2, 2)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'breakpoints' = [2]
--> Atribut: 'interpolation' = [2]
--> Atribut: 'type' = b'Tabulated1D'

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 2
--> Seluruh Data   : [[6.69771e+06 2.00000e+07]
 [1.00000e+00 1.00000e+00]]


Dalam konteks pustaka data OpenMC (seperti ENDF/B), atribut `'type' = b'Polynomial'` menandakan bahwa data tersebut **tidak disimpan sebagai tabel panjang berisi titik-titik diskrit** (seperti data *cross-section* yang Anda lihat sebelumnya), melainkan disimpan sebagai **rumus matematika**.

Secara spesifik, `Polynomial` berarti nilai fisisnya dihitung menggunakan persamaan polinomial (suku banyak) terhadap energi neutron datang ($E$).

### Membedah Data Anda

Path yang Anda bongkar adalah `Ac225/fission_energy_release/betas`. Ini merepresentasikan **rata-rata energi yang dilepaskan oleh partikel beta peluruhan** setelah terjadinya reaksi fisi, sebagai fungsi dari energi neutron yang menabrak target.

Bentuk umum persamaan polinomial di OpenMC adalah:

$$f(E) = C_0 + C_1 E + C_2 E^2 + \dots + C_n E^n$$

Di mana:

* $E$ = Energi neutron datang (dalam eV)
* $C_n$ = Koefisien polinomial

Dataset aktual yang Anda temukan memiliki 3 elemen: `[3.839037e+06, -7.500000e-02, 0.000000e+00]`. Angka-angka ini adalah koefisien dari rumus tersebut ($C_0, C_1, C_2$):

1. **$C_0$ (Konstanta):** $3.839037 \times 10^6$
2. **$C_1$ (Linear):** $-0.075$
3. **$C_2$ (Kuadratik):** $0$

### Makna Fisisnya

Karena nilai $C_2 = 0$, persamaan ini sebenarnya hanyalah garis lurus (linear). Jika kita masukkan koefisien tersebut ke dalam rumus, OpenMC akan menghitung energi beta peluruhan ($E_{\beta}$) menggunakan rumus berikut:

$$E_{\beta}(E) = 3.839037 \times 10^6 - 0.075E$$

*(Hasilnya dalam satuan eV. Nilai $3.83 \times 10^6$ eV setara dengan sekitar 3.84 MeV).*

**Mengapa disimpan dalam bentuk Polinomial?**
Untuk parameter tertentu seperti pelepasan energi fisi (beta, gamma, neutrino), perubahannya terhadap energi neutron sangat halus dan terprediksi—seringkali linier. Menyimpan 3 angka koefisien ini jauh lebih hemat memori dan lebih cepat diproses oleh komputer dibandingkan menyimpan ratusan titik data pada grid energi.

In [17]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/fission_energy_release/betas'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/fission_energy_release/betas

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (3,)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'type' = b'Polynomial'

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 3
--> Seluruh Data   : [ 3.839037e+06 -7.500000e-02  0.000000e+00]


Nilai ini merepresentasikan energi termal rata-rata dari medium pada suhu tersebut (dalam kasus Anda, pada suhu $1200\text{ K}$). Karena nilai $k \times T$ untuk satu suhu spesifik hanyalah satu angka konstan tunggal, OpenMC menyimpannya sebagai objek skalar, bukan array panjang seperti cross-section

In [16]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/kTs/1200K'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
# C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL")
        
        # Cek apakah ukuran dataset kosong () yang menandakan ini data skalar
        if ds.shape == ():
            data_aktual = ds[()]
            print(f"--> Nilai Skalar : {data_aktual}")
            
        # Jika bukan skalar, berarti ini array/list
        else:
            data_aktual = ds[:]
            print(f"--> Total elemen : {len(data_aktual)}")
            
            if len(data_aktual) > 10:
                # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
                print(f"--> 5 Data Pertama : {data_aktual[:5]}")
                print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
            else:
                print(f"--> Seluruh Data   : {data_aktual}")

Membongkar Path: Ac225/kTs/1200K

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: ()
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Tidak ada atribut yang menempel pada dataset ini.

[3] NILAI DATA AKTUAL
--> Nilai Skalar : 0.10341


In [10]:
import h5py
import numpy as np

# Ganti dengan nama file HDF5 OpenMC Anda
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Masuk ke path dataset spesifik
    path = 'Ac225/fission_energy_release/delayed_neutrons'
    
    # Pastikan path tersebut ada di dalam file
    if path in f:
        ds = f[path]
        
        print("==================================================")
        print(f"Membongkar Path: {path}")
        print("==================================================")
        
        # A. MEMBACA PROPERTI DATASET
        print("\n[1] PROPERTI DATASET BASIS")
        print(f"--> Jenis Objek : {type(ds)}")       # Memastikan ini adalah objek Dataset
        print(f"--> Ukuran/Shape: {ds.shape}")       # Menampilkan panjang array (misal: (21,))
        print(f"--> Tipe Data   : {ds.dtype}")       # Tipe data fisis (misal: float64)
        
        # B. MEMBACA ATRIBUT (Ini tempat threshold_idx berada)
        print("\n[2] ATRIBUT DATASET (METADATA)")
        if len(ds.attrs) == 0:
            print("--> Tidak ada atribut yang menempel pada dataset ini.")
        else:
            # Melakukan perulangan untuk membaca SEMUA atribut secara otomatis
            for key, val in ds.attrs.items():
                print(f"--> Atribut: '{key}' = {val}")
        
        # C. MEMBACA NILAI DATA AKTUAL
        print("\n[3] NILAI DATA AKTUAL (CROSS SECTION)")
        # Tanda [:] digunakan untuk mengekstrak seluruh isi dataset HDF5 menjadi array NumPy
        data_aktual = ds[:] 
        
        print(f"--> Total elemen: {len(data_aktual)}")
        if len(data_aktual) > 10:
            # Jika datanya panjang, cetak sampel awal dan akhir saja agar rapi
            print(f"--> 5 Data Pertama : {data_aktual[:5]}")
            print(f"--> 5 Data Terakhir: {data_aktual[-5:]}")
        else:
            print(f"--> Seluruh Data   : {data_aktual}")
            
    else:
        print(f"Error: Path '{path}' tidak ditemukan di dalam file {filename}.")

Membongkar Path: Ac225/fission_energy_release/delayed_neutrons

[1] PROPERTI DATASET BASIS
--> Jenis Objek : <class 'h5py._hl.dataset.Dataset'>
--> Ukuran/Shape: (3,)
--> Tipe Data   : float64

[2] ATRIBUT DATASET (METADATA)
--> Atribut: 'type' = b'Polynomial'

[3] NILAI DATA AKTUAL (CROSS SECTION)
--> Total elemen: 3
--> Seluruh Data   : [ 2.248e+04 -5.420e-04  1.130e-11]


Secara spesifik untuk `reaction_017` pada suhu 294K berdasarkan data yang Anda bongkar, nilai `threshold_idx = 945` berarti **nilai cross-section ($xs$) pada indeks ke-0 berpasangan dengan nilai energi pada indeks ke-945** di dalam grid energi utama isotop `Ac225`.

Berikut adalah visualisasi bagaimana data tersebut dipetakan secara matematis di dalam OpenMC:

### Cara Membaca Pemetaannya:

Jika Anda memiliki array energi utama bernama `energy_grid` (yang panjangnya sekitar 900-an atau lebih), maka:

* Nilai $xs[0]$ (yaitu `0.0`) $\rightarrow$ berpasangan dengan `energy_grid[945]` *(Inilah titik energi ambang batas fisisnya)*
* Nilai $xs[1]$ (yaitu `0.0009225`) $\rightarrow$ berpasangan dengan `energy_grid[946]`
* Nilai $xs[2]$ (yaitu `0.00497209`) $\rightarrow$ berpasangan dengan `energy_grid[947]`
* ... dan seterusnya sampai data terakhir:
* Nilai $xs[24]$ (yaitu `1.87087`) $\rightarrow$ berpasangan dengan `energy_grid[945 + 24]` atau `energy_grid[969]`

### Mengapa Desain HDF5 OpenMC Seperti Ini?

Untuk reaksi ambang (seperti *(n,2n)* atau reaksi hamburan inelastis tingkat tinggi pada $Ac^{225}$), probabilitas terjadinya reaksi pada energi rendah (indeks $0$ sampai $944$) adalah murni **nol**.

Daripada membuang-buang memori untuk menyimpan 945 angka nol (`0.0, 0.0, 0.0, ...`) di dalam file `.h5`, OpenMC memotong semuanya (*truncation*) dan hanya menyimpan 25 angka yang bermakna fisis saja, lalu memberikan "catatan kaki" berupa `threshold_idx = 945`.

In [6]:
import h5py

# Ganti dengan nama file HDF5 Anda yang sebenarnya
filename = 'neutron/Ac225.h5' 

with h5py.File(filename, 'r') as f:
    # 1. Akses dataset xs
    xs_dataset = f['Ac225/reactions/reaction_017/294K/xs']
    
    # 2. Ambil nilai indeks threshold dari atribut
    # Gunakan .get() untuk menghindari error jika atribut tidak ada
    threshold_idx = xs_dataset.attrs.get('threshold_idx', 0) 
    print(f"Indeks Threshold: {threshold_idx}")
    
    # 3. (Opsional) Melihat energi ambang aktual (dalam eV)
    # Anda perlu mengakses grid energi yang berpasangan dengannya
    # Path energi di OpenMC biasanya ada di '/<isotop>/energy/<suhu>'
    energy_grid = f['Ac225/energy/294K'] 
    
    threshold_energy = energy_grid[threshold_idx]
    print(f"Energi Threshold (Fisis): {threshold_energy} eV")

Indeks Threshold: 945
Energi Threshold (Fisis): 12386100.0 eV


In [ ]:
nama_grup = 'Ac225/reactions/reaction_017/600K'
isi_folder = list(file_data[nama_grup].keys())

print(f"Isi di dalam folder {nama_grup}:")
print(isi_folder)

Isi di dalam folder Ac225/reactions/reaction_017/600K:
['xs']


In [ ]:
nama_dataset = 'Ac225/reactions/reaction_017/600K/xs'
data_list = file_data[nama_dataset][()].tolist()
print(data_list)
print(len(data_list))

[0.0, 0.0009224972, 0.00497209, 0.0510171, 0.1183766, 0.185736, 0.2821125, 0.378489, 0.495722, 0.612955, 0.712518, 0.812081, 0.9326105, 1.05314, 1.13875, 1.22436, 1.319235, 1.41411, 1.53592, 1.63591, 1.69214, 1.76384, 1.797061, 1.82248, 1.87087]
25


Simple Maxwellian fission spectrum represented as

math::
$$ f(E \rightarrow E') = \frac{\sqrt{E'}}{I} e^{-E'/\theta(E)} $$

Parameters

theta : openmc.data.Tabulated1D
Tabulated function of incident neutron energy
u : float
Constant introduced to define the proper upper limit for the final
particle energy such that :math:`0 \le E' \le E - U`

Attributes

theta : openmc.data.Tabulated1D
Tabulated function of incident neutron energy
u : float
Constant introduced to define the proper upper limit for the final
particle energy such that :math:$`0 \le E' \le E - U`$

# Photon

In [5]:
import h5py

file_data = h5py.File('photon/Ac.h5', 'r')

# Menampilkan seluruh hierarki/struktur di dalam file
print("Struktur isi file:")
file_data.visit(print)

Struktur isi file:
Ac
Ac/bremsstrahlung
Ac/bremsstrahlung/dcs
Ac/bremsstrahlung/electron_energy
Ac/bremsstrahlung/ionization_energy
Ac/bremsstrahlung/num_electrons
Ac/bremsstrahlung/photon_energy
Ac/coherent
Ac/coherent/anomalous_imag
Ac/coherent/anomalous_real
Ac/coherent/integrated_scattering_factor
Ac/coherent/scattering_factor
Ac/coherent/xs
Ac/compton_profiles
Ac/compton_profiles/J
Ac/compton_profiles/binding_energy
Ac/compton_profiles/num_electrons
Ac/compton_profiles/pz
Ac/energy
Ac/incoherent
Ac/incoherent/scattering_factor
Ac/incoherent/xs
Ac/pair_production_electron
Ac/pair_production_electron/xs
Ac/pair_production_nuclear
Ac/pair_production_nuclear/xs
Ac/photoelectric
Ac/photoelectric/xs
Ac/subshells
Ac/subshells/K
Ac/subshells/K/transitions
Ac/subshells/K/xs
Ac/subshells/L1
Ac/subshells/L1/transitions
Ac/subshells/L1/xs
Ac/subshells/L2
Ac/subshells/L2/transitions
Ac/subshells/L2/xs
Ac/subshells/L3
Ac/subshells/L3/transitions
Ac/subshells/L3/xs
Ac/subshells/M1
Ac/subshells/M